# Kite — Kaggle model team (GGUF + llama.cpp)

Notebook này chạy các vai trò LLM trong đội của Kite trên **Kaggle GPU T4 x2**. Mỗi lần chỉ có **một model** được nạp, phục vụ qua `llama-server` (API tương thích OpenAI) bên trong session. Đổi model = tắt server cũ (giải phóng VRAM) rồi nạp model mới.

**Lý do không dùng transformers + bitsandbytes như bản trước:**
- Checkpoint gốc 24–32B có dung lượng 48–65 GB, vượt mức đĩa thực tế của Kaggle (~60 GB scratch, `/kaggle/working` ~20 GB).
- Devstral Small 2 được phát hành dưới dạng FP8, mà T4 (Turing, sm_75) không có kernel FP8.
- Gemma 4 có lỗi tràn số khi chạy fp16 trên T4 (với transformers; bản GGUF qua llama.cpp cần kiểm tra lại ở bước đo).

GGUF 4-bit chỉ nặng khoảng 7–25 GB/model, và chia được qua 2 GPU. `llama-server` cũng chính là kiểu endpoint mà OpenCode dùng được làm provider.

**Kiểm tra trước khi chạy**
1. Notebook để **private**. Bật **GPU T4 x2** và **Internet**.
2. **Add Input**: dataset `kite-llama-bin` và các dataset `kite-gguf-*` của model cần dùng.
3. Không đưa code riêng tư, token hay secret vào ô `TASK`.
4. Chạy lần lượt từ trên xuống. Muốn đổi model thì sửa `TASK["model"]` và chạy lại từ ô **Select model**.
5. Xong việc thì chạy ô **Stop** rồi tắt session (Kaggle không tự tắt GPU một cách tin cậy).

| Model (`TASK["model"]`) | Vai dự kiến | Ghi chú |
|---|---|---|
| `qwen3.6` | Model chính: agent trưởng, coding, review, vision | Qwen3.6-35B-A3B, MoE ~3B active; đã đo 23,4 GB VRAM, 54–58 tok/s |
| `devstral` | Code chuyên sâu (dự phòng) | Devstral Small 2 24B, dense, chậm hơn |
| `gemma4-26b` | Ứng viên agent trưởng (tiếng Việt) và gợi ý nhanh | Gemma 4 26B-A4B, MoE ~4B active |
| `gemma4-12b` | Nhỏ, nhanh | Gemma 4 12B (bản QAT của Unsloth) |
| `qwen3-coder-next` | Ứng viên code mạnh nhất | Qwen3-Coder-Next 80B-A3B; 36 GB > 30 GB VRAM nên llama.cpp tự đưa phần expert không vừa GPU sang RAM (`--fit`) |
| `qwen3-coder-30b` | Dự phòng cho code | Qwen3-Coder 30B-A3B |

Model nào làm loại task nào là do bộ điều phối trên EC2 quyết định (bảng điểm `routing.model_skill`), không phải notebook này. Model mới chỉ được dùng tự động sau khi đo (bước B).

**Model nằm trong dataset Kaggle (private), không tải từ Hugging Face khi chạy.** Mỗi model là một dataset `kite-gguf-<model>`, tạo một lần bằng một notebook uploader riêng trên Kaggle (chạy CPU, không tốn quota GPU). Gắn các dataset này vào notebook qua **Add Input**; `llama-server` đọc thẳng file GGUF trong `/kaggle/input`, không chép ra scratch. Model nào chưa gắn dataset thì worker báo lỗi rõ ràng khi được giao.

**Embedding (BGE-M3 / Nomic) không nằm trong notebook này.** Kaggle không chạy được 24/7: mỗi session tối đa 12 giờ, GPU có quota hằng tuần, và Kite tắt worker khi rảnh.

## 1. Kiểm tra GPU và đĩa

In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

# Kite điền dòng dưới khi tự bật notebook (`kite worker start`, hoặc tự động khi có việc). Chạy tay: để trống.
KITE_LAUNCH = {}
# Mở tay notebook mà Kite đã đẩy lên: chạy như chạy tay (KITE_LAUNCH chỉ dành cho lần Kite tự bật).
if KITE_LAUNCH and (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Interactive"
                    or time.time() > KITE_LAUNCH.get("expires", float("inf"))):
    KITE_LAUNCH = {}

assert shutil.which("nvidia-smi"), "Chưa bật GPU. Vào Settings -> Accelerator -> GPU T4 x2."
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

# Model lưu ở scratch (không bị giới hạn 20 GB của /kaggle/working, mất khi session tắt).
WORK = Path("/kaggle/tmp/kite")
MODELS_DIR = WORK / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK / "hf")          # cache HF cũng nằm trên scratch
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def free_gb(path=WORK):
    return shutil.disk_usage(path).free / 1024**3

print(f"Free on {WORK}: {free_gb():.1f} GiB")
print(f"Free on /kaggle/working: {shutil.disk_usage('/kaggle/working').free / 1024**3:.1f} GiB")

## 2. Build `llama-server` (CUDA, sm_75)

Lần đầu mất khoảng 5–15 phút. Script chỉ build target `llama-server` cho kiến trúc T4. Commit được dùng sẽ được in ra để ghi lại.

Muốn khỏi build lại: sau lần đầu, copy `/kaggle/tmp/kite/llama.cpp/build/bin` thành một Kaggle Dataset private, rồi trỏ `PREBUILT_BIN` vào đó.

In [ ]:
LLAMA_CPP_REF = "9048bad"   # commit đã build thành công trên Kaggle (chỉ dùng khi phải build lại)
USE_PREBUILT = True         # False = bỏ qua dataset, build lại từ nguồn

import glob

def find_libcuda():
    """Tìm libcuda.so cho CMake (target CUDA::cuda_driver); chỉ cần khi build."""
    for pattern in ("/usr/local/cuda/lib64/stubs/libcuda.so",
                    "/usr/local/cuda/targets/*/lib/stubs/libcuda.so",
                    "/usr/local/cuda*/lib64/stubs/libcuda.so"):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
    real = [line.split("=>")[-1].strip() for line in out.splitlines() if "libcuda.so" in line]
    real += sorted(glob.glob("/usr/lib*/**/libcuda.so*", recursive=True))
    real += sorted(glob.glob("/usr/local/nvidia/lib*/libcuda.so*"))
    real = [r for r in real if Path(r).exists()]
    if not real:
        raise FileNotFoundError("Không tìm thấy libcuda.so. Kiểm tra lại đã bật GPU chưa.")
    link_dir = WORK / "cuda-driver"
    link_dir.mkdir(parents=True, exist_ok=True)
    link = link_dir / "libcuda.so"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(real[0])
    return str(link)

LLAMA_DIR = WORK / "llama.cpp"
prebuilt = sorted(glob.glob("/kaggle/input/**/llama-server", recursive=True)) if USE_PREBUILT else []

if prebuilt:
    # /kaggle/input chỉ đọc và có thể mất quyền chạy: chép ra scratch rồi cấp quyền.
    local_bin = WORK / "llama-bin"
    shutil.rmtree(local_bin, ignore_errors=True)
    shutil.copytree(Path(prebuilt[0]).parent, local_bin)
    for f in local_bin.iterdir():
        f.chmod(0o755)
    SERVER_BIN = local_bin / "llama-server"
    LLAMA_COMMIT = f"prebuilt-{LLAMA_CPP_REF}"
    print("Dùng bản build sẵn từ", Path(prebuilt[0]).parent)
else:
    SERVER_BIN = LLAMA_DIR / "build" / "bin" / "llama-server"
    if not SERVER_BIN.exists():
        nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
        assert Path(nvcc).exists(), "Không tìm thấy nvcc trong image Kaggle."
        os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
        print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1])
        libcuda = find_libcuda()
        print("libcuda cho CMake:", libcuda)
        if not LLAMA_DIR.exists():
            subprocess.run(["git", "clone", "https://github.com/ggml-org/llama.cpp", str(LLAMA_DIR)], check=True)
        subprocess.run(["git", "-C", str(LLAMA_DIR), "checkout", LLAMA_CPP_REF], check=True)
        shutil.rmtree(LLAMA_DIR / "build", ignore_errors=True)
        subprocess.run(["cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
                        "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                        f"-DCMAKE_CUDA_COMPILER={nvcc}",
                        f"-DCUDA_cuda_driver_LIBRARY={libcuda}",
                        "-DLLAMA_CURL=OFF", "-DGGML_CCACHE=OFF",
                        "-DLLAMA_BUILD_TESTS=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
        subprocess.run(["cmake", "--build", str(LLAMA_DIR / "build"), "--config", "Release",
                        "-j", str(os.cpu_count() or 4), "--target", "llama-server"], check=True)
    LLAMA_COMMIT = subprocess.run(["git", "-C", str(LLAMA_DIR), "rev-parse", "--short", "HEAD"],
                                  capture_output=True, text=True).stdout.strip()

print("llama-server:", SERVER_BIN, "| llama.cpp commit:", LLAMA_COMMIT)

## 3. Roster, task và vị trí model

`ctx` theo số đo ô 5b (2026-11-19): 128K cho `gemma4-26b` và `qwen3.6` (vẫn nhanh, VRAM dư); 64K cho `devstral`, `qwen3-coder-30b`, `gemma4-12b` (ngữ cảnh dài làm chúng chậm hẳn); 32K cho `qwen3-coder-next` (đã không vừa VRAM). Lượt đầu OpenCode đã gửi khoảng 7.500 token system prompt + định nghĩa tool. KV cache nén `q8_0` (`KV_CACHE`) chỉ tốn nửa VRAM so với f16, chất lượng gần như không đổi. **Ngữ cảnh ở đây phải khớp `gateway.model_context` trong config Kite trên EC2.** Đo VRAM/tốc độ ở nhiều mức ngữ cảnh: ô *5b*.

`dataset` là slug của dataset Kaggle chứa file GGUF của model. `quant` phải khớp **đúng** tên biến thể (`Q4_K_M` khác `UD-Q4_K_M`). `extra_args` là cờ thêm cho `llama-server` của riêng model đó.

**Speculative decoding** (`SPEC_DECODING`): llama.cpp đoán trước nhiều token rồi model kiểm lại cả cụm trong một lượt; token nào sai thì model tự sinh lại, nên câu trả lời vẫn đúng như khi không bật, chỉ nhanh hơn. Mặc định dùng `ngram-mod` (không cần model phụ, nhanh khi model chép lại code đã có trong ngữ cảnh). Model có bản GGUF kèm đầu MTP (Multi-Token Prediction) thì dùng MTP: gắn dataset `mtp_dataset` của model đó (ví dụ `kite-gguf-qwen3-6-mtp` tạo từ `unsloth/Qwen3.6-35B-A3B-MTP-GGUF`, cùng quant) là worker tự dùng, không phải sửa config Kite. Nếu bản `llama-server` không nhận cờ, ô 5 tự chạy lại không có speculative decoding.

In [ ]:
import re

# Nén KV cache (bộ nhớ ngữ cảnh): q8_0 = nửa VRAM của f16. Đặt "f16" để tắt.
KV_CACHE = "q8_0"

# Speculative decoding (xem ghi chú ở trên): kết quả như khi không bật, chỉ nhanh hơn. False = tắt hẳn.
SPEC_DECODING = True
# ngram-mod: không cần model phụ. MoE cần draft dài, nên dùng bộ thông số mẫu cho MoE trong docs/speculative.md.
SPEC_NGRAM = ["--spec-type", "ngram-mod", "--spec-ngram-mod-n-match", "24",
              "--spec-ngram-mod-n-min", "48", "--spec-ngram-mod-n-max", "64"]
# MTP: bản GGUF có đầu MTP. n-max 6 + p-min 0.75 theo khuyến nghị của Unsloth cho Qwen3.6 MTP.
SPEC_MTP = ["--spec-type", "draft-mtp", "--spec-draft-n-max", "6", "--spec-draft-p-min", "0.75"]
SPEC_ARGS = []      # do prepare_model() chọn cho model đang nạp

ROSTER = {
    "qwen3.6": {
        "dataset": "kite-gguf-qwen3-6",
        "repo": "unsloth/Qwen3.6-35B-A3B-GGUF", "quant": "UD-Q4_K_M",
        # Gắn dataset này (unsloth/Qwen3.6-35B-A3B-MTP-GGUF, file UD-Q4_K_M) thì dùng nó thay bản trên, kèm MTP.
        "mtp_dataset": "kite-gguf-qwen3-6-mtp",
        "mmproj": True, "ctx": 131072, "max_tokens": 4096,
        # Khuyến nghị của dòng Qwen3 cho chế độ thinking; đo lại trên task thật rồi chỉnh.
        "sampling": {"temperature": 0.6, "top_p": 0.95, "top_k": 20},
        "system_in_user": False, "strip_think": True,
    },
    "devstral": {
        "dataset": "kite-gguf-devstral",
        "repo": "unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF", "quant": "Q4_K_M",
        "mmproj": False, "ctx": 65536, "max_tokens": 4096,
        "sampling": {"temperature": 0.15}, "system_in_user": False, "strip_think": False,
    },
    "gemma4-26b": {
        "dataset": "kite-gguf-gemma4-26b",
        "repo": "unsloth/gemma-4-26B-A4B-it-GGUF", "quant": "UD-Q4_K_M",
        "mmproj": False, "ctx": 131072, "max_tokens": 4096,
        "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": 64},
        "system_in_user": False, "strip_think": True,
    },
    "gemma4-12b": {
        "dataset": "kite-gguf-gemma4-12b",
        "repo": "unsloth/gemma-4-12b-it-qat-GGUF", "quant": "UD-Q4_K_XL",
        "mmproj": False, "ctx": 65536, "max_tokens": 4096,
        "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": 64},
        "system_in_user": False, "strip_think": True,
    },
    "qwen3-coder-next": {
        "dataset": "kite-gguf-qwen3-coder-next",
        # 33,8 GiB > 30 GiB VRAM: không ép -ngl, để llama.cpp tự chia (--fit): expert MoE không vừa GPU nằm ở RAM.
        "repo": "unsloth/Qwen3-Coder-Next-GGUF", "quant": "UD-Q3_K_XL",
        "mmproj": False, "ctx": 32768, "max_tokens": 4096,
        "sampling": {"temperature": 1.0, "top_p": 0.95, "top_k": 40},
        "system_in_user": False, "strip_think": False,
        "auto_fit": True,
        # Nạp hẳn vào RAM: mmap từ /kaggle/input khiến phần expert ở CPU đọc lại từ ổ dataset (đo được 0,6 tok/s).
        "extra_args": ["--load-mode", "none"],
    },
    "qwen3-coder-30b": {
        "dataset": "kite-gguf-qwen3-coder-30b",
        "repo": "unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF", "quant": "Q4_K_M",
        "mmproj": False, "ctx": 65536, "max_tokens": 4096,
        "sampling": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
        "system_in_user": False, "strip_think": False,
    },
}

# Router trên AWS sẽ điền phần này khi có transport được hỗ trợ; hiện tại nhập tay.
TASK = {
    "task_id": "manual-001",
    "model": "qwen3.6",
    "system_prompt": "You are a careful coding assistant. Give a concise, correct answer.",
    "prompt": "Write a Python function unique_preserving_order(items) that removes duplicates while preserving order, plus 3 pytest tests.",
    "image_path": None,   # vision: đường dẫn trong /kaggle/input/...
    "max_tokens": None,   # None = mặc định của model
}

INPUT = Path("/kaggle/input")


def ram_gb():
    info = dict(line.split(":", 1) for line in Path("/proc/meminfo").read_text().splitlines())
    kb = lambda key: int(info[key].split()[0])
    return kb("MemTotal") / 1024**2, kb("MemAvailable") / 1024**2


def quant_matches(filename, quant):
    """`...-Q4_K_M.gguf` hoặc shard `...-Q4_K_M-00001-of-00002.gguf`; `Q4_K_M` không khớp `UD-Q4_K_M`."""
    name = re.sub(r"-\d{5}-of-\d{5}(?=\.gguf$)", "", Path(filename).name.lower())
    if not name.endswith(".gguf") or "mmproj" in name:
        return False
    stem, q = name[:-5], quant.lower()
    return stem.endswith("-" + q) and (q.startswith("ud-") or not stem.endswith("-ud-" + q))


def dataset_dir(name, mtp=None):
    """Thư mục dataset của model đã gắn (Add Input), ưu tiên bản MTP nếu có; None nếu chưa gắn.
    mtp=False: chỉ bản thường; mtp=True: chỉ bản MTP."""
    spec = ROSTER[name]
    slugs = ([spec["mtp_dataset"]] if spec.get("mtp_dataset") and mtp is not False else [])
    slugs += [spec["dataset"]] if mtp is not True else []
    for slug in slugs:
        hits = sorted(p for p in INPUT.glob(f"**/{slug}") if p.is_dir())
        if hits:
            return hits[0]
    return None


def prepare_model(name):
    """Tìm file GGUF của model `name` trong dataset đã gắn; trả về (model_file, mmproj_file). Không tải gì."""
    spec, root = ROSTER[name], dataset_dir(name)
    if root is None:
        raise FileNotFoundError(f"Chưa gắn dataset `{spec['dataset']}` cho {name}: "
                                f"Add Input → Your Work → Datasets → {spec['dataset']} (tạo bằng notebook uploader).")
    weights = sorted(p for p in root.rglob("*.gguf") if quant_matches(p.name, spec["quant"]))
    if not weights:
        raise FileNotFoundError(f"{root} không có file {spec['quant']}. Có: "
                                + ", ".join(p.name for p in root.rglob("*.gguf")))
    mmproj = None
    if spec["mmproj"]:
        # Bản MTP có thể không kèm mmproj: lấy của bản thường nếu cũng đã gắn.
        roots = [root] + [r for r in [dataset_dir(name, mtp=False)] if r is not None and r != root]
        projs = sorted((p for r in roots for p in r.rglob("*mmproj*.gguf")),
                       key=lambda p: (p.name.lower() != "mmproj-f16.gguf", p.stat().st_size))
        if not projs:
            raise FileNotFoundError(f"{root} không có file mmproj cho vision.")
        mmproj = projs[0]
    global SPEC_ARGS
    has_mtp = bool(spec.get("mtp_dataset")) and root.name == spec["mtp_dataset"]
    default = SPEC_MTP if has_mtp else SPEC_NGRAM
    SPEC_ARGS = list(spec.get("spec_args", default)) if SPEC_DECODING else []
    size = sum(p.stat().st_size for p in weights) / 1024**3
    print(f"{name}: {weights[0].name} ({size:.1f} GiB, {len(weights)} file) từ {root}"
          f" | speculative decoding: {SPEC_ARGS[1] if SPEC_ARGS else 'tắt'}")
    return weights[0], mmproj


SPEC = ROSTER[TASK["model"]]
total, available = ram_gb()
print("Model:", TASK["model"], "->", SPEC["repo"], SPEC["quant"])
print(f"RAM: {available:.1f} / {total:.1f} GiB trống")
print("Dataset model đã gắn:", [n for n in ROSTER if dataset_dir(n)] or "chưa có")

## 4. Select model: lấy file từ dataset

Ô này tìm file GGUF của `TASK["model"]` trong dataset đã gắn. Chưa gắn dataset hoặc sai quant thì ô dừng với thông báo rõ ràng. Không tải gì từ Internet.

In [ ]:
TASK["model"] = KITE_LAUNCH.get("model") or TASK["model"]   # Kite tự bật: nạp luôn model đang được gọi
SPEC = ROSTER[TASK["model"]]
MODEL_FILE, MMPROJ_FILE = prepare_model(TASK["model"])
print(MODEL_FILE.name, "|", MMPROJ_FILE.name if MMPROJ_FILE else "no mmproj")

## 5. Khởi động `llama-server` (chỉ loopback)

Server bind vào `127.0.0.1`. **Không mở tunnel/public URL** để AWS gọi vào, đúng với gate an toàn của Kite. Model được chia layer qua 2 GPU; model có `auto_fit` để llama.cpp tự đưa phần expert MoE không vừa GPU sang RAM CPU.

In [ ]:
import requests

PORT = 8080
BASE = f"http://127.0.0.1:{PORT}"
LOG = WORK / "llama-server.log"


def stop_server():
    proc = globals().get("SERVER_PROC")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()


def launch(name, model_file, mmproj_file, spec_args=None):
    """Khởi động llama-server cho `name` và chờ tới khi sẵn sàng; trả về số giây nạp."""
    global SERVER_PROC, SPEC_ARGS
    spec = ROSTER[name]
    spec_args = SPEC_ARGS if spec_args is None else spec_args
    stop_server()
    # --parallel 1: giữ trọn ctx 32K cho một request (OpenCode gửi ~7.500 token ngay lượt đầu).
    # auto_fit: không đặt -ngl, vì --fit (tự chia GPU/CPU theo bộ nhớ trống) chỉ chạy khi -ngl không bị ép.
    gpu = [] if spec.get("auto_fit") else ["-ngl", "999"]
    cmd = [str(SERVER_BIN), "-m", str(model_file), "--host", "127.0.0.1", "--port", str(PORT),
           *gpu, "--split-mode", "layer", "-c", str(spec["ctx"]), "--parallel", "1",
           # KV cache nén (V) cần flash attention: bật hẳn thay vì "auto".
           "--flash-attn", "on" if KV_CACHE != "f16" else "auto",
           "--cache-type-k", KV_CACHE, "--cache-type-v", KV_CACHE,
           "--jinja", "--no-webui", *map(str, spec.get("extra_args", [])), *map(str, spec_args)]
    if mmproj_file:
        cmd += ["--mmproj", str(mmproj_file)]
    # Bản build dùng thư viện .so nằm cạnh file chạy (libllama, libggml...): chỉ đường cho loader.
    env = dict(os.environ, LD_LIBRARY_PATH=f"{SERVER_BIN.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}")
    SERVER_PROC = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env)
    t0 = time.monotonic()
    while True:
        if SERVER_PROC.poll() is not None:
            if spec_args:
                # Bản llama-server này không nhận cờ speculative (hoặc thiếu VRAM cho phần draft): chạy lại không có nó.
                print("speculative decoding không khởi động được, chạy lại không có nó:\n" + LOG.read_text()[-1500:])
                SPEC_ARGS = []
                return launch(name, model_file, mmproj_file, spec_args=[])
            print(LOG.read_text()[-4000:])
            raise RuntimeError("llama-server thoát sớm. Xem log ở trên (thường do OOM: với model có "
                               "--n-cpu-moe thì tăng số đó; hoặc thiếu thư viện .so / flag không hỗ trợ).")
        try:
            if requests.get(f"{BASE}/health", timeout=2).status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.monotonic() - t0 > 900:
            raise TimeoutError("Model chưa sẵn sàng sau 15 phút.")
        time.sleep(3)
    seconds = round(time.monotonic() - t0, 1)
    total, available = ram_gb()
    print(f"{name} sẵn sàng sau {seconds}s | RAM trống {available:.1f}/{total:.1f} GiB"
          f" | speculative decoding: {' '.join(spec_args[:2]) if spec_args else 'tắt'}")
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
    if spec.get("auto_fit"):
        print("\n".join(line for line in LOG.read_text().splitlines() if "fit" in line.lower())[-3000:])
    return seconds


LOAD_SECONDS = launch(TASK["model"], MODEL_FILE, MMPROJ_FILE)

## 5b. Đo ngữ cảnh (chạy tay, tùy chọn)

Nạp từng model ở nhiều mức `ctx`, ghi VRAM, rồi gửi một prompt dài ~70% ngữ cảnh để đo tốc độ đọc prompt thật. Mất khoảng 30–40 phút. Đặt `RUN_CONTEXT_PROBE = True` rồi chạy ô này **khi worker (ô 8) chưa chạy**. Bản Save & Run All bỏ qua ô này.

Chọn mức lớn nhất mà `ok`, VRAM trống ≥ 1 GiB và tốc độ đọc prompt còn chấp nhận được; ghi vào `ROSTER` (ô 3) và `gateway.model_context` trên EC2.


In [ ]:
RUN_CONTEXT_PROBE = False
PROBE_MODELS = ["gemma4-26b", "qwen3.6", "devstral", "qwen3-coder-30b"]
PROBE_CTX = [65536, 98304, 131072]
PROBE_FILL = 0.7          # prompt dài bằng 70% ngữ cảnh


def vram_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    rows = [[int(x) for x in line.split(",")] for line in out.strip().splitlines()]
    return sum(r[0] for r in rows), sum(r[1] for r in rows)


def probe(name, ctx):
    spec = ROSTER[name]
    old_ctx = spec["ctx"]
    spec["ctx"] = ctx
    try:
        model_file, mmproj_file = prepare_model(name)
        try:
            load_s = launch(name, model_file, mmproj_file)
        except Exception as exc:          # thường là hết VRAM: mức này quá lớn
            return {"model": name, "ctx": ctx, "ok": False, "error": str(exc)[:120]}
        used, total = vram_mib()
        # Khoảng 12 token mỗi lặp; con số thật lấy từ timings.prompt_n.
        filler = "def add_one(x):\n    return x + 1\n\n" * int(ctx * PROBE_FILL / 12)
        body = {"messages": [{"role": "user", "content": filler + "Tóm tắt đoạn code trên trong một câu."}],
                "max_tokens": 32, "temperature": 0, "chat_template_kwargs": {"enable_thinking": False}}
        started = time.monotonic()
        reply = requests.post(f"{BASE}/v1/chat/completions", json=body, timeout=3600)
        timings = reply.json().get("timings", {}) if reply.ok else {}
        return {"model": name, "ctx": ctx, "ok": reply.ok, "load_s": load_s,
                "vram_used_gib": round(used / 1024, 1), "vram_free_gib": round((total - used) / 1024, 1),
                "prompt_tokens": timings.get("prompt_n"), "prompt_s": round(time.monotonic() - started, 1),
                "prompt_tps": round(timings.get("prompt_per_second", 0)),
                "gen_tps": round(timings.get("predicted_per_second", 0), 1),
                "error": None if reply.ok else reply.text[:120]}
    finally:
        spec["ctx"] = old_ctx
        stop_server()


if RUN_CONTEXT_PROBE:
    PROBE_RESULTS = []
    for name in PROBE_MODELS:
        for ctx in PROBE_CTX:
            row = probe(name, ctx)
            PROBE_RESULTS.append(row)
            print(json.dumps(row, ensure_ascii=False))
            if not row["ok"]:
                break                     # mức lớn hơn chắc chắn cũng không vừa
    print("\nĐề xuất (lớn nhất mà ok và VRAM trống >= 1 GiB):")
    for name in PROBE_MODELS:
        good = [r for r in PROBE_RESULTS if r["model"] == name and r["ok"] and r.get("vram_free_gib", 0) >= 1]
        print(f"  {name}: {max(r['ctx'] for r in good) if good else 'giữ 65536'}")
else:
    print("Bỏ qua đo ngữ cảnh (RUN_CONTEXT_PROBE = False).")


## 6. Chạy task

Có thể gọi `chat()` nhiều lần trong cùng session (multi-turn), mỗi lần là một request tới server đang chạy.

In [ ]:
import base64, re

def build_messages(task, spec):
    user_content = task["prompt"]
    if spec["system_in_user"] and task.get("system_prompt"):
        user_content = task["system_prompt"].strip() + "\n\n" + user_content
    if task.get("image_path"):
        if not spec["mmproj"]:
            raise ValueError("Model này không có mmproj cho ảnh; dùng qwen3.6.")
        img = Path(task["image_path"])
        if not img.is_file():
            raise FileNotFoundError(img)
        mime = "image/png" if img.suffix.lower() == ".png" else "image/jpeg"
        data = base64.b64encode(img.read_bytes()).decode()
        user_content = [{"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}},
                        {"type": "text", "text": user_content}]
    messages = []
    if task.get("system_prompt") and not spec["system_in_user"]:
        messages.append({"role": "system", "content": task["system_prompt"]})
    messages.append({"role": "user", "content": user_content})
    return messages

def chat(messages, spec, max_tokens=None):
    body = {"messages": messages, "max_tokens": max_tokens or spec["max_tokens"], **spec["sampling"]}
    r = requests.post(f"{BASE}/v1/chat/completions", json=body, timeout=1800)
    r.raise_for_status()
    return r.json()

if KITE_LAUNCH:
    print("Kite tự bật notebook: bỏ qua task mẫu.")
else:
    started = time.monotonic()
    resp = chat(build_messages(TASK, SPEC), SPEC, TASK.get("max_tokens"))
    elapsed_seconds = round(time.monotonic() - started, 2)

    choice = resp["choices"][0]
    raw_text = choice["message"].get("content") or ""
    reasoning = choice["message"].get("reasoning_content") or ""
    # Một số template trả <think> ngay trong content; tách ra để kết quả gọn.
    m = re.search(r"<think>(.*?)</think>", raw_text, flags=re.S)
    if m:
        reasoning = reasoning or m.group(1).strip()
        raw_text = raw_text[m.end():]
    result_text = raw_text.strip() if SPEC["strip_think"] else (choice["message"].get("content") or "").strip()

    usage = resp.get("usage", {})
    print(result_text)
    print("\n---")
    print("finish_reason:", choice.get("finish_reason"), "| usage:", usage, "| seconds:", elapsed_seconds)
    if choice.get("finish_reason") == "length":
        print("Cảnh báo: bị cắt do max_tokens. Tăng TASK['max_tokens'] rồi chạy lại ô này.")

## 7. Export kết quả

Đây là phong bì batch chạy tay, **không phải** `WorkerResult` có lease của Kite. Nó không sửa repo và không push.

In [ ]:
if not KITE_LAUNCH:
    response = {
        "schema_version": 1,
        "task_id": TASK["task_id"],
        "model": TASK["model"],
        "model_repo": SPEC["repo"],
        "model_file": MODEL_FILE.name,
        "quant": SPEC["quant"],
        "runtime": {"engine": "llama.cpp", "commit": LLAMA_COMMIT, "ctx": SPEC["ctx"],
                    "sampling": SPEC["sampling"], "load_seconds": LOAD_SECONDS},
        "status": "truncated" if choice.get("finish_reason") == "length" else "completed",
        "text": result_text,
        "reasoning_chars": len(reasoning),
        "usage": {"input_tokens": usage.get("prompt_tokens", 0), "output_tokens": usage.get("completion_tokens", 0)},
        "elapsed_seconds": elapsed_seconds,
    }
    OUTPUT_PATH = Path("/kaggle/working/kite_worker_result.json")
    OUTPUT_PATH.write_text(json.dumps(response, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", OUTPUT_PATH)

## 8. Worker mode (nối EC2)

Notebook thành một worker của Kite: tự nhận việc từ gateway trên EC2, tự đổi model khi cần, chạy cho tới khi bấm **Interrupt** (■) hoặc gần hết 12 giờ session.

Trước khi chạy:
1. **Add-ons → Secrets**: thêm secret `KITE_WORKER_TOKEN` (token in ra từ `kite worker token` trên EC2) và bật nó cho notebook này.
2. Sửa `GATEWAY_URL` ở ô thứ ba thành `gateway.public_url` trên EC2.

Khi đang ở worker mode, ô cuối sẽ chạy mãi. Đó là bình thường: nó đang chờ việc.

**Kite tự bật (không cần mở notebook):** khi config Kite có `launch` cho worker này, Kite đẩy notebook này lên Kaggle như một version mới ("Save & Run All", GPU T4 x2) mỗi khi có việc mà chưa có worker, rồi điền `KITE_LAUNCH` ở ô 1. Lúc đó notebook nạp luôn model đang được gọi, bỏ qua task mẫu, và tự tắt sau `idle_minutes` phút không có việc. Dataset và cài đặt của notebook trên Kaggle được giữ nguyên.

In [ ]:
"""Pull worker: runs next to llama-server (for example in a Kaggle notebook) and serves the gateway.

Standard library only, so it can be pasted into a notebook cell without installing anything.

Loop: send a heartbeat, long-poll ``GET /worker/jobs``, make sure the requested model is loaded,
forward the request to the local llama-server, post the answer back. A background thread keeps
heartbeating while a long request runs, so the gateway does not think the worker died.

When the client asked for streaming, the answer is read from llama-server as it is generated and
its chunks go to the gateway about twice a second (``POST /worker/jobs/<id>/chunks``); if the
gateway no longer wants them (the client went away), generation stops.
"""

from __future__ import annotations

import json
import threading
import time
import urllib.error
import urllib.request
from typing import Any, Callable


class PullWorker:
    def __init__(
        self,
        gateway_url: str,
        token: str,
        *,
        ensure_model: Callable[[str], None],
        loaded_model: Callable[[], str | None],
        cached_models: Callable[[], list[str]] = lambda: [],
        llama_url: str = "http://127.0.0.1:8080",
        poll_wait_seconds: int = 50,
        heartbeat_seconds: float = 20.0,
        request_timeout_seconds: float = 1800.0,
        stop_after_seconds: float | None = None,
        stop_when_idle_seconds: float | None = None,
        stream_flush_seconds: float = 0.5,
        log: Callable[[str], None] = print,
    ) -> None:
        self.gateway_url = gateway_url.rstrip("/")
        self.token = token
        self.ensure_model = ensure_model
        self.loaded_model = loaded_model
        self.cached_models = cached_models
        self.llama_url = llama_url.rstrip("/")
        self.poll_wait_seconds = poll_wait_seconds
        self.heartbeat_seconds = heartbeat_seconds
        self.request_timeout_seconds = request_timeout_seconds
        self.deadline = time.monotonic() + stop_after_seconds if stop_after_seconds else None
        # Started by Kite (not by hand): stop after this long without a job so the GPU quota is not
        # spent on an idle session. Kite starts the notebook again when work arrives.
        self.stop_when_idle_seconds = stop_when_idle_seconds
        self.last_job_at = time.monotonic()
        self.stream_flush_seconds = stream_flush_seconds
        self.log = log
        self.ready = True
        self.draining = False
        self._stop = threading.Event()
        self.jobs_done = 0

    # ------------------------------------------------------------------ HTTP
    def _call(self, method: str, url: str, body: Any = None, *, timeout: float = 70,
              auth: bool = True) -> tuple[int, Any]:
        data = json.dumps(body).encode("utf-8") if body is not None else None
        headers = {"Content-Type": "application/json"}
        if auth:
            headers["Authorization"] = f"Bearer {self.token}"
        request = urllib.request.Request(url, data=data, method=method, headers=headers)
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                raw = response.read()
                return response.status, (json.loads(raw) if raw else None)
        except urllib.error.HTTPError as exc:
            raw = exc.read()
            try:
                return exc.code, json.loads(raw) if raw else None
            except json.JSONDecodeError:
                return exc.code, raw.decode("utf-8", "replace")

    def heartbeat(self) -> None:
        status, body = self._call("POST", f"{self.gateway_url}/worker/heartbeat", {
            "ready": self.ready and not self._stop.is_set(),
            "draining": self.draining,
            "loaded_model": self.loaded_model(),
            "cached_models": self.cached_models(),
        }, timeout=30)
        if status == 401:
            raise PermissionError("gateway rejected the worker token")
        if status != 200:
            raise RuntimeError(f"heartbeat failed: HTTP {status} {body}")
        if isinstance(body, dict) and body.get("stop") and not self._stop.is_set():
            # `kite worker stop` on EC2: finish the current job, then end the session.
            self.log("Gateway yêu cầu dừng worker (kite worker stop).")
            self.draining = True
            self._stop.set()

    def _heartbeat_loop(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            try:
                self.heartbeat()
            except PermissionError:
                self.log("Token bị gateway từ chối; dừng worker.")
                self._stop.set()
            except Exception as exc:  # network blips must not kill the worker
                self.log(f"heartbeat lỗi (sẽ thử lại): {exc}")

    # ------------------------------------------------------------------ jobs
    def run_job(self, job: dict[str, Any]) -> None:
        job_id, model = job["job_id"], job["model"]
        started = time.monotonic()
        try:
            if self.loaded_model() != model:
                self.log(f"[{job_id[:8]}] nạp model {model}...")
                self.ready = False
                self._safe_heartbeat()
                self.ensure_model(model)
                self.ready = True
                self._safe_heartbeat()
            if job.get("stream"):
                payload = {"status": "ok", **self._stream(job_id, job["body"])}
            else:
                status, response = self._call("POST", f"{self.llama_url}/v1/chat/completions", job["body"],
                                              timeout=self.request_timeout_seconds, auth=False)
                if status != 200 or not isinstance(response, dict):
                    raise RuntimeError(f"llama-server HTTP {status}: {str(response)[:500]}")
                payload = {"status": "ok", "response": response}
        except Exception as exc:
            self.ready = True
            payload = {"status": "error", "error": f"{type(exc).__name__}: {exc}"[:2000]}
        status, body = self._call("POST", f"{self.gateway_url}/worker/jobs/{job_id}/result", payload)
        seconds = time.monotonic() - started
        self.jobs_done += 1
        self.last_job_at = time.monotonic()
        self.log(f"[{job_id[:8]}] {model} {payload['status']} sau {seconds:.1f}s (gateway: HTTP {status})")

    def _stream(self, job_id: str, body: dict[str, Any]) -> dict[str, Any]:
        """Stream the answer from llama-server, passing its chunks to the gateway as they come.

        Returns the whole answer assembled as a normal (non-streamed) completion, for the gateway's
        journal, plus the chunks the gateway has not acknowledged yet (``start`` = index of the first).
        """
        pending: list[dict] = []            # chunks the gateway has not acknowledged yet
        state = {"sent": 0, "gone": False}
        lock = threading.Lock()
        finished = threading.Event()

        def flush() -> None:
            with lock:
                batch, start = list(pending), state["sent"]
            if not batch:
                return
            status, _ = self._call("POST", f"{self.gateway_url}/worker/jobs/{job_id}/chunks",
                                   {"start": start, "chunks": batch}, timeout=30)
            if status == 200:
                with lock:
                    del pending[:len(batch)]
                    state["sent"] = start + len(batch)
            elif status in (404, 409):
                state["gone"] = True        # the client went away or the job was given up

        def flusher() -> None:
            while not finished.wait(self.stream_flush_seconds):
                try:
                    flush()
                except Exception:           # a network blip: the chunks stay pending, sent next time
                    pass

        request = urllib.request.Request(
            f"{self.llama_url}/v1/chat/completions", method="POST",
            data=json.dumps({**body, "stream": True, "stream_options": {"include_usage": True}}).encode("utf-8"),
            headers={"Content-Type": "application/json"})
        chunks: list[dict] = []
        thread = threading.Thread(target=flusher, daemon=True)
        thread.start()
        try:
            try:
                response = urllib.request.urlopen(request, timeout=self.request_timeout_seconds)
            except urllib.error.HTTPError as exc:
                raise RuntimeError(f"llama-server HTTP {exc.code}: "
                                   f"{exc.read().decode('utf-8', 'replace')[:500]}") from exc
            with response:
                for raw in response:
                    if state["gone"]:
                        raise RuntimeError("the gateway no longer wants this answer (client gone)")
                    line = raw.decode("utf-8", "replace").strip()
                    if not line.startswith("data:"):
                        continue
                    data = line[len("data:"):].strip()
                    if data == "[DONE]":
                        break
                    chunk = json.loads(data)
                    if not isinstance(chunk, dict):
                        continue
                    if "error" in chunk:
                        raise RuntimeError(f"llama-server: {str(chunk['error'])[:500]}")
                    chunks.append(chunk)
                    with lock:
                        pending.append(chunk)
        finally:
            finished.set()
            thread.join()
        with lock:
            return {"response": assemble_stream(chunks), "start": state["sent"], "chunks": list(pending)}

    def _safe_heartbeat(self) -> None:
        try:
            self.heartbeat()
        except PermissionError:
            raise
        except Exception as exc:
            self.log(f"heartbeat lỗi: {exc}")

    def stop(self) -> None:
        self._stop.set()

    def run(self) -> None:
        self.heartbeat()   # fail fast on a wrong URL or token
        self.log(f"Worker đã kết nối {self.gateway_url}, model đang nạp: {self.loaded_model()}")
        beat = threading.Thread(target=self._heartbeat_loop, daemon=True)
        beat.start()
        backoff = 1.0
        try:
            while not self._stop.is_set():
                if self.deadline is not None and time.monotonic() > self.deadline:
                    self.log("Hết thời gian session dự kiến: chuyển sang draining và dừng.")
                    self.draining = True
                    self._safe_heartbeat()
                    break
                idle = time.monotonic() - self.last_job_at
                if self.stop_when_idle_seconds and idle > self.stop_when_idle_seconds:
                    self.log(f"Không có việc {idle / 60:.0f} phút: tự tắt để đỡ tốn quota GPU "
                             "(Kite sẽ tự bật lại khi có việc).")
                    self.draining = True
                    self._safe_heartbeat()
                    break
                try:
                    status, body = self._call(
                        "GET", f"{self.gateway_url}/worker/jobs?wait={self.poll_wait_seconds}",
                        timeout=self.poll_wait_seconds + 20)
                except (urllib.error.URLError, TimeoutError, ConnectionError) as exc:
                    self.log(f"Mất kết nối gateway ({exc}); thử lại sau {backoff:.0f}s")
                    self._stop.wait(backoff)
                    backoff = min(backoff * 2, 60)
                    continue
                backoff = 1.0
                if status == 204:
                    continue
                if status == 401:
                    raise PermissionError("gateway rejected the worker token")
                if status != 200 or not isinstance(body, dict):
                    self.log(f"GET /worker/jobs trả HTTP {status}: {body}")
                    self._stop.wait(5)
                    continue
                self.run_job(body)
        finally:
            self.ready = False
            self._stop.set()
            try:
                self.heartbeat_after_stop()
            except Exception:
                pass

    def heartbeat_after_stop(self) -> None:
        self._call("POST", f"{self.gateway_url}/worker/heartbeat", {
            "ready": False, "draining": True, "loaded_model": self.loaded_model(),
            "cached_models": self.cached_models()}, timeout=10)


def assemble_stream(chunks: list[dict[str, Any]]) -> dict[str, Any]:
    """The chunks of a streamed chat completion, put back together as one non-streamed completion."""
    content: list[str] = []
    reasoning: list[str] = []
    calls: dict[int, dict[str, Any]] = {}
    finish, usage, timings, first = None, None, None, (chunks[0] if chunks else {})
    for chunk in chunks:
        usage = chunk.get("usage") or usage
        timings = chunk.get("timings") or timings
        for choice in chunk.get("choices") or []:
            delta = choice.get("delta") or {}
            if delta.get("content"):
                content.append(delta["content"])
            if delta.get("reasoning_content"):
                reasoning.append(delta["reasoning_content"])
            for call in delta.get("tool_calls") or []:
                slot = calls.setdefault(call.get("index", 0), {
                    "id": None, "type": "function", "function": {"name": "", "arguments": ""}})
                function = call.get("function") or {}
                slot["id"] = slot["id"] or call.get("id")
                if function.get("name") and not slot["function"]["name"]:
                    slot["function"]["name"] = function["name"]
                slot["function"]["arguments"] += function.get("arguments") or ""
            finish = choice.get("finish_reason") or finish
    message: dict[str, Any] = {"role": "assistant", "content": "".join(content) or None}
    if reasoning:
        message["reasoning_content"] = "".join(reasoning)
    if calls:
        message["tool_calls"] = [{**calls[i], "id": calls[i]["id"] or f"call_{i}"} for i in sorted(calls)]
    response: dict[str, Any] = {
        "id": first.get("id") or "chatcmpl-kite", "object": "chat.completion",
        "created": first.get("created") or int(time.time()), "model": first.get("model") or "",
        "choices": [{"index": 0, "message": message, "finish_reason": finish or "stop"}]}
    if usage:
        response["usage"] = usage
    if timings:
        response["timings"] = timings
    return response


In [ ]:
from kaggle_secrets import UserSecretsClient

GATEWAY_URL = KITE_LAUNCH.get("gateway_url") or "https://184-72-143-157.sslip.io:8443"   # gateway.public_url trên EC2
SESSION_HOURS = 11.5                       # Kaggle cắt session ở 12 giờ: ngừng nhận việc trước đó
# Kite tự bật: hết việc lâu thì tự tắt cho đỡ tốn quota GPU (Kite bật lại khi có việc). Chạy tay: chạy tới khi bấm dừng.
IDLE_MINUTES = KITE_LAUNCH.get("idle_minutes")

WORKER_TOKEN = KITE_LAUNCH.get("token")   # chỉ có khi config Kite đặt launch.token: per-launch
if not WORKER_TOKEN:
    try:
        WORKER_TOKEN = UserSecretsClient().get_secret("KITE_WORKER_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Không đọc được Secret KITE_WORKER_TOKEN. Chạy tay: Add-ons -> Secrets, bật KITE_WORKER_TOKEN cho "
            "notebook này. Kite tự bật: Kaggle có thể không gắn Secrets cho bản đẩy qua API; khi đó đặt "
            "launch.token: per-launch trong config Kite trên EC2.") from exc


def start_llama(name):
    """Đổi sang model `name`: tắt server cũ (giải phóng VRAM), nạp model mới từ dataset đã gắn."""
    global SPEC, MODEL_FILE, MMPROJ_FILE, LOAD_SECONDS
    stop_server()
    MODEL_FILE, MMPROJ_FILE = prepare_model(name)
    SPEC = ROSTER[name]
    LOAD_SECONDS = launch(name, MODEL_FILE, MMPROJ_FILE)
    TASK["model"] = name


def loaded_model():
    proc = globals().get("SERVER_PROC")
    return TASK["model"] if proc is not None and proc.poll() is None else None


def cached_models():
    return [name for name in ROSTER if dataset_dir(name)]


worker = PullWorker(GATEWAY_URL, WORKER_TOKEN, ensure_model=start_llama, loaded_model=loaded_model,
                    cached_models=cached_models, stop_after_seconds=SESSION_HOURS * 3600,
                    stop_when_idle_seconds=IDLE_MINUTES * 60 if IDLE_MINUTES else None)
try:
    worker.run()
except KeyboardInterrupt:
    worker.stop()
    print(f"Đã dừng worker sau {worker.jobs_done} việc.")

## 9. Stop

Tắt server và xóa model trên scratch. Sau đó **tắt session** trong giao diện Kaggle (Run → Stop session) để không tốn quota GPU.

In [ ]:
stop_server()
shutil.rmtree(MODELS_DIR, ignore_errors=True)
print("Server stopped, model files removed. Now stop the Kaggle session from the UI.")